# 5. Bayesian Optimisation for DFT Parameters

Replace the standard 80-point grid search for (ENCUT, KPPRA) convergence
with Bayesian Optimisation that finds the Pareto-optimal settings in ~12 evaluations.

**Algorithm:**
- Surrogate: Gaussian Process (Matern-5/2 kernel)
- Acquisition: Expected Improvement + ParEGO cost scalarization
- Output: Pareto front of (accuracy, cost) trade-offs

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 5.1 Define a synthetic energy landscape
We simulate a realistic VASP-like energy convergence surface.

In [ ]:
def simulated_vasp_energy(encut, kppra, n_atoms=36):
    """
    Simulated DFT energy that converges with increasing ENCUT and KPPRA.
    E(ENCUT,KPPRA) = E_converged + A*exp(-ENCUT/200) + B*exp(-KPPRA/600) + noise
    """
    E_converged = -245.678  # eV (typical slab)
    basis_error = 0.8 * np.exp(-encut / 200)
    kpoint_error = 0.3 * np.exp(-kppra / 600)
    noise = np.random.default_rng(int(encut * 100 + kppra)).normal(0, 0.001)
    return E_converged + basis_error + kpoint_error + noise

# Visualise the energy landscape
encuts = np.arange(300, 601, 20)
kppras = np.arange(400, 3201, 200)
E_grid = np.zeros((len(encuts), len(kppras)))
for i, e in enumerate(encuts):
    for j, k in enumerate(kppras):
        E_grid[i, j] = simulated_vasp_energy(e, k)

E_ref = simulated_vasp_energy(600, 3200)
error_grid = np.abs(E_grid - E_ref) / 36  # eV/atom

plt.figure(figsize=(8, 5))
c = plt.contourf(kppras, encuts, error_grid * 1000, levels=20, cmap='RdYlGn_r')
plt.colorbar(c, label='Energy error (meV/atom)')
plt.xlabel('KPPRA'); plt.ylabel('ENCUT (eV)')
plt.title('Energy convergence landscape (simulated)')
plt.show()

## 5.2 Run Bayesian Optimisation

In [ ]:
from science.optimization.bayesian_params import BayesianParameterOptimizer, dft_cost_model

opt = BayesianParameterOptimizer(
    n_atoms=36,
    encut_range=(300, 600),
    kppra_range=(400, 3200),
    target_error=0.001,  # 1 meV/atom convergence target
)

# Phase 1: Initial exploration (Latin hypercube)
initial_points = opt.suggest_initial(n=5)
print('Phase 1: Initial exploration')
for encut, kppra in initial_points:
    energy = simulated_vasp_energy(encut, kppra)
    opt.observe(encut, kppra, energy)
    print(f'  ENCUT={encut:4.0f}, KPPRA={kppra:4d} → E={energy:.4f} eV')

# Phase 2: BO loop
print('\nPhase 2: Bayesian optimisation')
for i in range(10):
    encut, kppra = opt.suggest_next()
    energy = simulated_vasp_energy(encut, kppra)
    opt.observe(encut, kppra, energy)
    error = opt.observations[-1].energy_error
    print(f'  Step {i+1:2d}: ENCUT={encut:4.0f}, KPPRA={kppra:4d} → '
          f'error={error*1000:.2f} meV/atom  cost={dft_cost_model(encut, kppra):.2f}x')

print(f'\n{opt.summary()}')

## 5.3 Visualise the BO trajectory

In [ ]:
result = opt.result()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Sampled points on convergence landscape
ax = axes[0]
ax.contourf(kppras, encuts, error_grid * 1000, levels=20, cmap='RdYlGn_r', alpha=0.4)
for i, p in enumerate(result.all_points):
    color = 'blue' if i < 5 else 'red'
    marker = 'o' if i < 5 else 's'
    ax.plot(p.kppra, p.encut, marker, color=color, markersize=8, alpha=0.8)
ax.plot(result.optimal_kppra, result.optimal_encut, '*', color='gold',
        markersize=15, markeredgecolor='black', label='Optimal')
ax.set_xlabel('KPPRA'); ax.set_ylabel('ENCUT (eV)')
ax.set_title(f'BO samples ({result.n_evaluations} evals)\nBlue=initial, Red=BO')
ax.legend()

# 2. Convergence history
ax = axes[1]
errors = [p.energy_error * 1000 for p in result.all_points]
ax.plot(errors, 'o-', color='steelblue')
ax.axhline(1.0, color='green', ls='--', label='1 meV/atom target')
ax.set_xlabel('Evaluation #'); ax.set_ylabel('Error (meV/atom)')
ax.set_title('Convergence history')
ax.legend()

# 3. Pareto front
ax = axes[2]
for p in result.all_points:
    ax.plot(p.cost, p.energy_error * 1000, 'o', color='lightgray', alpha=0.5)
pareto_costs = [p.cost for p in result.pareto_front]
pareto_errors = [p.energy_error * 1000 for p in result.pareto_front]
ax.plot(pareto_costs, pareto_errors, 's-', color='red', markersize=8, label='Pareto front')
ax.set_xlabel('Relative cost'); ax.set_ylabel('Error (meV/atom)')
ax.set_title('Pareto front: accuracy vs cost')
ax.legend()

plt.tight_layout()
plt.show()

print(f'\nOptimal: ENCUT={result.optimal_encut:.0f} eV, KPPRA={result.optimal_kppra}')
print(f'Error: {result.predicted_error*1000:.2f} meV/atom')
print(f'Savings: ~{max(0, 80 - result.n_evaluations)} DFT runs saved vs grid search')

## 5.4 GP Surrogate Prediction
Visualise the GP posterior over the parameter space.

In [ ]:
# Predict over the full grid using the trained GP
from science.optimization.bayesian_params import GaussianProcess

grid_points = np.array([[e, k] for e in encuts for k in kppras])
grid_norm = grid_points.copy().astype(float)
grid_norm[:, 0] = (grid_norm[:, 0] - 300) / 300
grid_norm[:, 1] = (grid_norm[:, 1] - 400) / 2800

mu, sigma = opt._gp.predict(grid_norm)
mu_grid = mu.reshape(len(encuts), len(kppras))
sigma_grid = sigma.reshape(len(encuts), len(kppras))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

c1 = ax1.contourf(kppras, encuts, mu_grid * 1000, levels=20, cmap='RdYlGn_r')
plt.colorbar(c1, ax=ax1, label='Predicted error (meV/atom)')
ax1.set_title('GP mean prediction'); ax1.set_xlabel('KPPRA'); ax1.set_ylabel('ENCUT')

c2 = ax2.contourf(kppras, encuts, sigma_grid * 1000, levels=20, cmap='Purples')
plt.colorbar(c2, ax=ax2, label='Uncertainty (meV/atom)')
ax2.set_title('GP uncertainty (explore here)'); ax2.set_xlabel('KPPRA'); ax2.set_ylabel('ENCUT')

# Mark observed points
for ax in [ax1, ax2]:
    for p in result.all_points:
        ax.plot(p.kppra, p.encut, 'kx', markersize=6)

plt.tight_layout()
plt.show()